# Extraction and source predictors: categorical hierarchies

This focused notebook completes the initial **categorical hierarchy** review of
`extraction_type`, `extraction_type_group`, `extraction_type_class`, `source`, `source_type`, `source_class`. It describes the supplied training and test predictors,
then uses the labelled training rows to identify relationships worth
carrying into a leakage-safe modelling pipeline.

Two three-level hierarchies describe how water is extracted and where it originates.

This is exploratory evidence, not fitted preprocessing. Category pooling,
imputation, encoding and scaling must be learned inside each training fold.


## Consistent audit contract

Every focused predictor audit answers the same questions before adding
type-specific checks:

1. What is explicitly missing, and what looks like a sentinel?
2. What range or category coverage is present in training and test?
3. How much of the test set is exposed to unseen training levels?
4. Does the labelled distribution vary enough to justify retaining the field?
5. What exact baseline treatment follows from the evidence?

Target-rate tables flag support rather than treating tiny groups as reliable.
Train/test comparisons are descriptive and do not use the hidden test labels.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

stage_directory = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "data" / "TrainingSetValues.csv").is_file()
)
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    MISSING_CATEGORY,
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    cramer_v,
    hierarchy_conflicts,
    hierarchy_summary,
    numeric_summary,
    numeric_target_summary,
    normalise_categories,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)
audited_features = ['extraction_type', 'extraction_type_group', 'extraction_type_class', 'source', 'source_type', 'source_class']
assert set(audited_features).issubset(training_features.columns)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {len(audited_features)} predictors."
)


Validated 59,400 training rows and 14,850 test rows for 6 predictors.


## 1. Missingness, cardinality and test coverage

Source blanks, pandas nulls and configured sentinel strings are reported separately.
Semantic sentinels such as `unknown` stay visible in frequency and target tables;
they are not silently merged with blank values.
Rare means fewer than 50 training rows; it is a diagnostic threshold, not
a preprocessing choice. Total-variation distance compares marginal shares.


In [2]:
category_overview = categorical_summary(
    training_features,
    test_features,
    audited_features,
    rare_threshold=50,
    sentinel_tokens_by_column={},
)
display(category_overview)


,training explicit missing,training source blank rows,training sentinel rows,test explicit missing,test source blank rows,test sentinel rows,training levels,test levels,training levels with <50 rows,training rows in rare levels (%),test-only levels,test rows in unseen levels (%),training-only levels,marginal total-variation distance
feature,,,,,,,,,,,,,,
extraction_type,0,0,0,0,0,0,18,17,3,0.14,0,0.0,1,0.0154
extraction_type_group,0,0,0,0,0,0,13,13,0,0.00,0,0.0,0,0.0154
extraction_type_class,0,0,0,0,0,0,7,7,0,0.00,0,0.0,0,0.0143
source,0,0,66,0,0,20,10,10,0,0.00,0,0.0,0,0.0100
source_type,0,0,0,0,0,0,7,7,0,0.00,0,0.0,0,0.0088
source_class,0,0,278,0,0,69,3,3,0,0.00,0,0.0,0,0.0029


## 2. Most common values


In [3]:
for feature in audited_features:
    print()
    print(feature)
    display(category_frequency_table(training_features, test_features, feature, top_n=10))



extraction_type


,training rows,training (%),test rows,test (%)
extraction_type,,,,
gravity,26780,45.08,6483,43.66
nira/tanira,8154,13.73,2051,13.81
other,6430,10.82,1672,11.26
submersible,4764,8.02,1218,8.2
swn 80,3670,6.18,918,6.18
mono,2865,4.82,763,5.14
india mark ii,2400,4.04,629,4.24
afridev,1770,2.98,438,2.95
ksb,1415,2.38,375,2.53



extraction_type_group


,training rows,training (%),test rows,test (%)
extraction_type_group,,,,
gravity,26780,45.08,6483,43.66
nira/tanira,8154,13.73,2051,13.81
other,6430,10.82,1672,11.26
submersible,6179,10.4,1593,10.73
swn 80,3670,6.18,918,6.18
mono,2865,4.82,763,5.14
india mark ii,2400,4.04,629,4.24
afridev,1770,2.98,438,2.95
rope pump,451,0.76,121,0.81



extraction_type_class


...

,training rows,training (%),test rows,test (%)
source,,,,
spring,17021,28.65,4195,28.25
shallow well,16824,28.32,4316,29.06
machine dbh,11075,18.64,2747,18.5
river,9612,16.18,2352,15.84
rainwater harvesting,2295,3.86,568,3.82
hand dtw,874,1.47,234,1.58
lake,765,1.29,185,1.25
dam,656,1.1,184,1.24
other,212,0.36,49,0.33



source_type


,training rows,training (%),test rows,test (%)
source_type,,,,
spring,17021,28.65,4195,28.25
shallow well,16824,28.32,4316,29.06
borehole,11949,20.12,2981,20.07
river/lake,10377,17.47,2537,17.08
rainwater harvesting,2295,3.86,568,3.82
dam,656,1.1,184,1.24
other,278,0.47,69,0.46



source_class


,training rows,training (%),test rows,test (%)
source_class,,,,
groundwater,45794,77.09,11492,77.39
surface,13328,22.44,3289,22.15
unknown,278,0.47,69,0.46


## 3. Relationship with `status_group`

The tables display the most supported levels first and mark whether each
level has at least 150 training rows. Small groups are leads
for later validation, not stable target encodings.


In [4]:
for feature in audited_features:
    print()
    print(feature)
    profile = categorical_target_profile(
        training_data,
        feature,
        minimum_support=150,
    )
    display(profile.head(15))



extraction_type


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
extraction_type,,,,,
gravity,26780,True,59.93,10.09,29.99
nira/tanira,8154,True,66.48,7.86,25.66
other,6430,True,16.00,3.20,80.79
submersible,4764,True,55.12,4.76,40.11
swn 80,3670,True,56.95,5.78,37.28
mono,2865,True,37.77,4.50,57.73
india mark ii,2400,True,60.33,3.29,36.38
afridev,1770,True,67.80,2.37,29.83
ksb,1415,True,49.68,1.84,48.48



extraction_type_group


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
extraction_type_group,,,,,
gravity,26780,True,59.93,10.09,29.99
nira/tanira,8154,True,66.48,7.86,25.66
other,6430,True,16.00,3.20,80.79
submersible,6179,True,53.88,4.09,42.03
swn 80,3670,True,56.95,5.78,37.28
mono,2865,True,37.77,4.50,57.73
india mark ii,2400,True,60.33,3.29,36.38
afridev,1770,True,67.80,2.37,29.83
rope pump,451,True,64.97,3.77,31.26



extraction_type_class


...

status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
source,,,,,
spring,17021,True,62.23,7.50,30.27
shallow well,16824,True,49.48,5.69,44.83
machine dbh,11075,True,48.96,4.43,46.61
river,9612,True,56.86,12.70,30.44
rainwater harvesting,2295,True,60.39,13.68,25.93
hand dtw,874,True,56.86,1.95,41.19
lake,765,True,21.18,1.57,77.25
dam,656,True,38.57,3.66,57.77
other,212,True,59.43,0.47,40.09



source_type


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
source_type,,,,,
spring,17021,True,62.23,7.50,30.27
shallow well,16824,True,49.48,5.69,44.83
borehole,11949,True,49.54,4.25,46.21
river/lake,10377,True,54.23,11.88,33.89
rainwater harvesting,2295,True,60.39,13.68,25.93
dam,656,True,38.57,3.66,57.77
other,278,True,56.83,1.80,41.37



source_class


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
source_class,,,,,
groundwater,45794,True,54.23,5.99,39.78
surface,13328,True,54.52,11.79,33.70
unknown,278,True,56.83,1.80,41.37


## 4. Related-field consistency

A deterministic child-to-parent mapping makes the parent derivable from
the child in this dataset. That is redundancy evidence, not automatic
permission to discard the child: granularity, unseen levels and model
behaviour still determine which representation is safer.


In [5]:
hierarchy_relationships = [('extraction_type', 'extraction_type_group'), ('extraction_type_group', 'extraction_type_class'), ('source', 'source_type'), ('source_type', 'source_class')]
display(
    hierarchy_summary(
        training_features,
        test_features,
        hierarchy_relationships,
    )
)
for child, parent in hierarchy_relationships:
    conflicts = hierarchy_conflicts(training_features, child, parent)
    if not conflicts.empty:
        print()
        print(f"Training conflicts for {child} -> {parent}")
        display(conflicts)


complete rows  \
relationship                                   frame                     
extraction_type -> extraction_type_group       training          59400   
                                               test              14850   
extraction_type_group -> extraction_type_class training          59400   
                                               test              14850   
source -> source_type                          training          59400   
                                               test              14850   
source_type -> source_class                    training          59400   
                                               test              14850   

                                                         child levels  \
relationship                                   frame                    
extraction_type -> extraction_type_group       training            18   
                                               test                17   
extraction_type_group -> extraction_type_class training            13   
                                               test                13   
source -> source_type                          training            10   
                                               test                10   
source_type -> source_class                    training             7   
                                               test                 7   

                                                         parent levels  \
relationship                                   frame                     
extraction_type -> extraction_type_group       training             13   
                                               test                 13   
extraction_type_group -> extraction_type_class training              7   
                                               test                  7   
source -> source_type                          training              7   
                                               test                  7   
source_type -> source_class                    training              3   
                                               test                  3   

                                                         ambiguous child levels  \
relationship                                   frame                              
extraction_type -> extraction_type_group       training                       0   
                                               test                           0   
extraction_type_group -> extraction_type_class training                       0   
                                               test                           0   
source -> source_type                          training                       0   
                                               test                           0   
source_type -> source_class                    training                       0   
                                               test                           0   

                                                         rows in ambiguous child levels  \
relationship                                   frame                                      
extraction_type -> extraction_type_group       training                               0   
                                               test                                   0   
extraction_type_group -> extraction_type_class training                               0   
                                               test                                   0   
source -> source_type                          training                               0   
                                               test                                   0   
source_type -> source_class                    training                               0   
                                               test                                   0   

                                                         deterministic child-to-parent  \
relationship

## Training/test handoff


In [6]:
display(
    category_overview[[
        "training levels",
        "test levels",
        "test-only levels",
        "test rows in unseen levels (%)",
        "training rows in rare levels (%)",
        "marginal total-variation distance",
    ]].sort_values("test rows in unseen levels (%)", ascending=False)
)


,training levels,test levels,test-only levels,test rows in unseen levels (%),training rows in rare levels (%),marginal total-variation distance
feature,,,,,,
extraction_type,18,17,0,0.0,0.14,0.0154
extraction_type_group,13,13,0,0.0,0.00,0.0154
extraction_type_class,7,7,0,0.0,0.00,0.0143
source,10,10,0,0.0,0.00,0.0100
source_type,7,7,0,0.0,0.00,0.0088
source_class,3,3,0,0.0,0.00,0.0029


## Decision register

The register separates observed evidence from the proposed baseline action.
A retained field is still a candidate: later validation must show whether it
improves generalisation and whether a coarser related representation is safer.


In [7]:
decision_register = pd.DataFrame([{'feature': 'extraction_type', 'quality finding': 'Eighteen levels map deterministically upward; `other` is 80.79% non-functional.', 'baseline treatment': 'Keep the granular type; rare-pool if validation requires it.', 'risk to verify': 'Related hierarchy levels are redundant when used together without evidence.'}, {'feature': 'extraction_type_group', 'quality finding': 'Deterministic from type and only 0.0013 descriptive MI bits lower.', 'baseline treatment': 'Treat as a strong redundancy candidate beside extraction_type.', 'risk to verify': 'Related hierarchy levels are redundant when used together without evidence.'}, {'feature': 'extraction_type_class', 'quality finding': 'Seven deterministic broad classes give a compact back-off.', 'baseline treatment': 'Compare type plus class against type alone; do not keep all three.', 'risk to verify': 'Related hierarchy levels are redundant when used together without evidence.'}, {'feature': 'source', 'quality finding': 'Ten fully test-covered levels retain lake/river differences hidden by source_type.', 'baseline treatment': 'Keep granular source and compare with the intermediate type.', 'risk to verify': 'Related hierarchy levels are redundant when used together without evidence.'}, {'feature': 'source_type', 'quality finding': 'Deterministic from source and removes useful within-type detail.', 'baseline treatment': 'Use only as a lower-cardinality ablation candidate.', 'risk to verify': 'Related hierarchy levels are redundant when used together without evidence.'}, {'feature': 'source_class', 'quality finding': 'Three deterministic classes have weak descriptive association.', 'baseline treatment': 'First source-hierarchy level to omit from baseline.', 'risk to verify': 'Related hierarchy levels are redundant when used together without evidence.'}])
display(decision_register.set_index("feature"))


,quality finding,baseline treatment,risk to verify
feature,,,
extraction_type,Eighteen levels map deterministically upward; `other` is 80.79% non-functional.,Keep the granular type; rare-pool if validation requires it.,Related hierarchy levels are redundant when used together without evidence.
extraction_type_group,Deterministic from type and only 0.0013 descriptive MI bits lower.,Treat as a strong redundancy candidate beside extraction_type.,Related hierarchy levels are redundant when used together without evidence.
extraction_type_class,Seven deterministic broad classes give a compact back-off.,Compare type plus class against type alone; do not keep all three.,Related hierarchy levels are redundant when used together without evidence.
source,Ten fully test-covered levels retain lake/river differences hidden by source...,Keep granular source and compare with the intermediate type.,Related hierarchy levels are redundant when used together without evidence.
source_type,Deterministic from source and removes useful within-type detail.,Use only as a lower-cardinality ablation candidate.,Related hierarchy levels are redundant when used together without evidence.
source_class,Three deterministic classes have weak descriptive association.,First source-hierarchy level to omit from baseline.,Related hierarchy levels are redundant when used together without evidence.


### Handoff to modelling

Do not one-hot every level of a deterministic hierarchy by default. Compare granular and coarse alternatives on the same split.

- Preserve raw source frames and implement the stated sentinel rules on copies.
- Fit imputers, rare-level grouping and encoders on each training fold only.
- Map unseen validation or test categories to an explicit fallback.
- Compare the stated baseline treatment with a simple omission ablation.
- Revisit target-rate observations after the reproducible stratified split exists.
